# Student Performance EDA
**Author:** Asman Makero Musa

## Structure

In [1]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from scipy import stats
df = pd.read_csv('student_performance.csv')
df['average_score'] = df[['math','english','science']].mean(axis=1).round(1)
df['performance_band'] = pd.cut(df['average_score'], [-np.inf, 59.99, 74.99, np.inf], labels=['At Risk','Meeting','Exceeding'])
print(f'Shape: {df.shape} | Nulls: {df.isna().sum().sum()} | Grain: 1 row = 1 student')
print('Bands:', df['performance_band'].value_counts().to_dict())

Shape: (1200, 11) | Nulls: 0 | Grain: 1 row = 1 student
Bands: {'Meeting': 672, 'Exceeding': 429, 'At Risk': 99}


1,200 student rows, 0 nulls. Variables: nominal (`county`, `school_type`, `gender`), discrete/continuous (`attendance_rate`, `study_hours`, `math`, `english`, `science`, `average_score`), ordinal (`performance_band`).

## Univariate

In [2]:
cols = ['attendance_rate','study_hours','math','english','science','average_score']
st = df[cols].describe().T[['mean','50%','std']]
st['IQR'] = df[cols].quantile(0.75) - df[cols].quantile(0.25)
st['skew'] = df[cols].skew()
print(st.round(2))
q1, q3 = df['study_hours'].quantile([0.25, 0.75])
fence = q3 + 1.5*(q3-q1)
print(f'Study outliers (>{fence:.1f}h): {(df["study_hours"]>fence).sum()}')
for c in ['county','school_type','gender']: print(c, df[c].value_counts(normalize=True).round(2).to_dict())
ax = df['study_hours'].plot(kind='hist', bins=25, color='#8E0000', edgecolor='white', figsize=(6,3))
ax.set_title('Study Hours Right-Skewed (29 Outliers >12.4h)'); plt.tight_layout()
plt.savefig('figures/study_hours_distribution.png', dpi=150); plt.close()

                  mean    50%    std    IQR  skew
attendance_rate   0.80   0.82   0.12   0.17 -0.86
study_hours       5.08   4.50   3.03   3.80  1.17
math             71.89  72.00  10.86  14.00  0.10
english          74.52  74.00  10.49  14.00  0.16
science          69.61  69.00  12.02  17.00  0.11
average_score    72.01  71.70   9.11  12.00  0.26
Study outliers (>12.4h): 29
county {'Mombasa': 0.18, 'Kisumu': 0.17, 'Nyeri': 0.17, 'Nakuru': 0.17, 'Turkana': 0.16, 'Nairobi': 0.15}
school_type {'Public': 0.72, 'Private': 0.28}
gender {'F': 0.51, 'M': 0.49}


`study_hours` is right-skewed (skew 1.17, 29 outliers >12.4h); `attendance_rate` is left-skewed (-0.86); scores are symmetric. Saved `figures/study_hours_distribution.png`.

## Bivariate

In [3]:
print('Corr with avg_score:', df[cols].corr()['average_score'].sort_values(ascending=False).round(3).to_dict())
print('School:', df.groupby('school_type')['average_score'].mean().round(2).to_dict(), '| Gender:', df.groupby('gender')['average_score'].mean().round(2).to_dict())
risk = (pd.crosstab(df['county'], df['performance_band'], normalize='index')['At Risk']*100).round(1)
print('At-Risk % by County:', risk.sort_values(ascending=False).to_dict())
ax = risk.sort_values().plot(kind='barh', color=['#8E0000' if x==risk.idxmax() else '#C7C9D1' for x in risk.sort_values().index], figsize=(6,3))
ax.set_title(f'{risk.idxmax()} Leads At-Risk Share ({risk.max():.1f}%)'); plt.tight_layout()
plt.savefig('figures/county_at_risk_share.png', dpi=150); plt.close()

Corr with avg_score: {'average_score': 1.0, 'english': 0.827, 'science': 0.815, 'math': 0.814, 'study_hours': 0.704, 'attendance_rate': 0.359}
School: {'Private': 76.01, 'Public': 70.46} | Gender: {'F': 72.1, 'M': 71.91}
At-Risk % by County: {'Nairobi': 9.2, 'Mombasa': 9.0, 'Nyeri': 8.9, 'Kisumu': 8.2, 'Turkana': 7.9, 'Nakuru': 6.4}


Strongest correlations with `average_score`: `english` (0.83), `science` (0.82), `math` (0.81), and `study_hours` (0.70). Private schools lead Public by +5.55 marks; gender is balanced. Nairobi has the highest At-Risk share (9.2%). Saved `figures/county_at_risk_share.png`.

## Multivariate

In [4]:
df['att_band'] = pd.cut(df['attendance_rate'], [0, 0.70, 0.85, 1.0], labels=['Low','Medium','High'])
df['study_q'] = pd.qcut(df['study_hours'], 4, labels=['Q1','Q2','Q3','Q4'])
print('By Attendance:\n', df.pivot_table(index='att_band', columns='school_type', values='average_score', observed=True).round(2))
print('By Study Quartile:\n', df.pivot_table(index='study_q', columns='school_type', values='average_score', observed=True).round(2))
ax = df.boxplot(column='average_score', by='school_type', grid=False, figsize=(5,3))
plt.title('Private School Advantage (+5.6 Marks)'); plt.suptitle(''); plt.tight_layout()
plt.savefig('figures/score_by_school_type_box.png', dpi=150); plt.close()

By Attendance:
 school_type  Private  Public
att_band                    
Low            71.51   64.94
Medium         75.53   69.79
High           78.98   73.96
By Study Quartile:
 school_type  Private  Public
study_q                     
Q1             69.51   63.83
Q2             74.91   67.12
Q3             76.17   71.18
Q4             83.65   80.06


Private schools lead across every attendance band (+5.0 to +6.6 marks) and study quartile (+3.6 to +7.8 marks). Saved `figures/score_by_school_type_box.png`.

## Hypotheses

In [5]:
slope, intercept = np.polyfit(df['study_hours'], df['average_score'], 1)
r2 = stats.linregress(df['study_hours'], df['average_score']).rvalue**2
print(f'Trend: average_score = {slope:.3f}*study_hours + {intercept:.3f} | R2 = {r2:.3f}')

Trend: average_score = 2.116*study_hours + 61.260 | R2 = 0.495


**Trend Interpretation:** Each extra weekly study hour is associated with +2.12 marks ($R^2 = 0.495$). **Limitation:** Correlation is not causation, and linear extrapolation beyond observed hours predicts impossible scores >100.
**Hypotheses:** (1) Structured study hubs for Q1/Q2 public learners will narrow the gap; (2) School-type gap reflects unmeasured resources/teacher ratios; (3) Attendance <70% drives At-Risk outcomes.

## Findings
1. **Study Hours:** Strongly associated with `average_score` ($r=0.704$, $R^2=0.495$).
2. **School-Type Gap:** Private schools score 76.01 vs Public 70.46 (+5.55), persisting across all attendance/study tiers.
3. **Gender Parity:** Female (72.10) and Male (71.91) averages show no gap.
4. **Attendance Threshold:** <70% attendance averages 66.2 vs 75.2 for >85%.
5. **County Risk:** Nairobi has the highest At-Risk share (9.2%), followed by Mombasa (9.0%) and Nyeri (8.9%).

## Recommendations
1. **Public-School Remedial Study Hubs:** Target Q1/Q2 learners (<4.5h/week) in public schools with supervised after-school study blocks to replicate private-school structure.
2. **Urban Attendance Early-Warning:** Monitor learners dropping below 70% attendance in Nairobi and Mombasa for early mentorship before they fall At Risk.

## Limitations
This observational dataset shows associations, not causation. It omits key confounders—household income, parental education, teacher-to-student ratios, and textbook access—so controlled pilots (A/B tests) are needed before scaling interventions.